In [1]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# Cek ketersediaan GPU agar training lebih cepat
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Menggunakan perangkat: {device}')

Menggunakan perangkat: cuda


In [2]:
# Konfigurasi Hyperparameter
batch_size = 64
latent_dim = 100  # Ukuran vektor noise (latent space)
epochs = 50  # Bisa diubah antara 50 - 100

# Transformasi: Ubah gambar ke tensor dan normalisasi ke rentang [-1, 1]
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
)

# Download dan muat dataset MNIST
dataloader = torch.utils.data.DataLoader(
    torchvision.datasets.MNIST(
        root='./data', train=True, transform=transform, download=True
    ),
    batch_size=batch_size,
    shuffle=True,
)

print(f'Jumlah batch per epoch: {len(dataloader)}')

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 503kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.63MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.5MB/s]

Jumlah batch per epoch: 938


In [3]:
# 1. GENERATOR: Dense + Transposed Convolution
class Generator(nn.Module):

  def __init__(self):
    super(Generator, self).__init__()
    self.fc = nn.Sequential(
        nn.Linear(latent_dim, 128 * 7 * 7),
        nn.BatchNorm1d(128 * 7 * 7),
        nn.ReLU(True),
    )
    self.conv_blocks = nn.Sequential(
        nn.Unflatten(1, (128, 7, 7)),
        nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # 14x14
        nn.BatchNorm2d(64),
        nn.ReLU(True),
        nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1),  # 28x28
        nn.Tanh(),  # Output rentang [-1, 1]
    )

  def forward(self, z):
    out = self.fc(z)
    out = self.conv_blocks(out)
    return out


# 2. DISCRIMINATOR: Konvolusional + Dense
class Discriminator(nn.Module):

  def __init__(self):
    super(Discriminator, self).__init__()
    self.conv_blocks = nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),  # 14x14
        nn.LeakyReLU(0.2, inplace=True),
        nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # 7x7
        nn.BatchNorm2d(64),
        nn.LeakyReLU(0.2, inplace=True),
        nn.Flatten(),
        nn.Linear(64 * 7 * 7, 1),
        nn.Sigmoid(),  # Output probabilitas (0 atau 1)
    )

  def forward(self, img):
    return self.conv_blocks(img)


# Inisialisasi Model, Loss, dan Optimizer
generator = Generator().to(device)
discriminator = Discriminator().to(device)

criterion = nn.BCELoss()
optimizer_G = torch.optim.Adam(
    generator.parameters(), lr=0.0002, betas=(0.5, 0.999)
)
optimizer_D = torch.optim.Adam(
    discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999)
)

# Noise tetap untuk memvisualisasikan perkembangan pada titik latent yang sama
fixed_noise = torch.randn(16, latent_dim, device=device)

In [ ]:
history_images = []

print('Memulai proses training GAN...')
for epoch in range(epochs):
  for i, (imgs, _) in enumerate(dataloader):
    # ---------------------
    #  Train Discriminator
    # ---------------------
    real_imgs = imgs.to(device)
    valid = torch.ones(real_imgs.size(0), 1, device=device)
    fake = torch.zeros(real_imgs.size(0), 1, device=device)

    # Loss data asli
    optimizer_D.zero_grad()
    real_loss = criterion(discriminator(real_imgs), valid)

    # Loss data palsu (generator)
    z = torch.randn(real_imgs.size(0), latent_dim, device=device)
    fake_imgs = generator(z)
    fake_loss = criterion(discriminator(fake_imgs.detach()), fake)

    d_loss = (real_loss + fake_loss) / 2
    d_loss.backward()
    optimizer_D.step()

    # -----------------
    #  Train Generator
    # -----------------
    optimizer_G.zero_grad()
    g_loss = criterion(discriminator(fake_imgs), valid)  # Ingin menipu D
    g_loss.backward()
    optimizer_G.step()

    # Print status tiap batch/epoch
    print(
        f'[Epoch {epoch+1}/{epochs}] [Batch {i}/{len(dataloader)}] '
        f'[D loss: {d_loss.item():.4f}] [G loss: {g_loss.item():.4f}]',
        end='\r',
    )

  # Simpan visualisasi setiap 10 epoch untuk melihat perbandingan bertahap
  if (epoch + 1) % 10 == 0 or epoch == 0:
    generator.eval()
    with torch.no_grad():
      sample_imgs = generator(fixed_noise).detach().cpu()
      history_images.append((epoch + 1, sample_imgs))
    generator.train()

print('\nTraining Selesai!')

Memulai proses training GAN...


In [ ]:
# Menampilkan evolusi hasil gambar dari generator dari awal hingga akhir epoch
fig, axes = plt.subplots(1, len(history_images), figsize=(15, 3))

for idx, (ep, img_tensor) in enumerate(history_images):
  # Gabungkan 16 gambar menjadi grid 4x4
  grid = torchvision.utils.make_grid(
      img_tensor, nrow=4, normalize=True
  ).permute(1, 2, 0)

  axes[idx].imshow(grid.numpy())
  axes[idx].axis('off')
  axes[idx].set_title(f'Epoch {ep}', fontsize=12)

plt.suptitle(
    'Perbandingan Hasil Generasi Gambar MNIST Bertahap (GAN)', fontsize=14
)
plt.tight_layout()
plt.show()